# Phase 2 — Boto3 Ingestion Utility (VS Code)

In [ ]:
# Install boto3 if needed
# %pip install boto3

## Local CLI Configuration (The Bridge to AWS)

**Why are we doing this?**
Right now, you are logged into AWS in your *web browser*. However, your *Python scripts* running inside VS Code are completely blind; they don't know who you are or what AWS account you own. 

Using the `aws configure` command creates a secure "VIP Pass" (authentication token) on your local machine so your Python scripts can securely interact with your cloud resources.

### Step 2: Get your Credentials
Login to AWS Console. Click on the **My Security Credentials** on the right of the page. Create the access key if not done. Copy the **AccessKeyID** and **SecretAccessKey** after creating it which will be needed to access your aws account through CLI

### Step 3: Set Up AWS CLI Access
To configure your credentials, you need to run the AWS configure command in your VS Code terminal. 
Since traditional copy-pasting into web terminals can be tricky, please carefully select and copy the text from the code cell below.

In [4]:
# 👇 Select and copy the command below, then paste it into your VS Code terminal 👇
# aws configure

When prompted in the terminal, enter your credentials exactly like this:
```text
AWS Access Key ID [None]: <Paste your AccessKeyID here>
AWS Secret Access Key [None]: <Paste your SecretAccessKey here>
Default region name [None]: us-east-1
Default output format [None]: json

In [1]:
import os
from dotenv import load_dotenv
import boto3
load_dotenv()
my_bucket_name = os.getenv("AWS_BUCKET_NAME")  # Get the bucket name from the .env file

print(my_bucket_name)

phase2-s3-pyboto3-bucket-mohank-1234


In [2]:
import boto3, json
from botocore.exceptions import ClientError

# Initialize our Boto3 Clients
iam = boto3.client("iam")
sts = boto3.client("sts")

policy_name = f"boto3-S3logs-mohank-Policy"
account_id = sts.get_caller_identity()["Account"]

# Define the IAM Policy Document (JSON rules for what this script is allowed to do)
policy_doc = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Action": ["s3:*", "logs:*"],
        "Resource": "*"
    }]
}

try:
    # Programmatically create the policy
    res = iam.create_policy(
        PolicyName=policy_name,
        PolicyDocument=json.dumps(policy_doc)
    )
    print("INFO: IAM Policy Created Successfully.")
    print("ARN:", res["Policy"]["Arn"])

except ClientError as e:
    if e.response["Error"]["Code"] == "EntityAlreadyExists":
        print("INFO: IAM Policy already exists:")
        print(f"ARN: arn:aws:iam::{account_id}:policy/{policy_name}")
    else:
        print("ERROR:", e)

INFO: IAM Policy already exists:
ARN: arn:aws:iam::983457875389:policy/boto3-S3logs-mohank-Policy


In [3]:
import boto3

# SOLUTION: Function to list customer-managed IAM policies
def list_custom_iam_policies():
    result = []
    
    # Initialize the IAM client
    iam = boto3.client('iam')
    
    # Call list_policies, filtering for 'Local' (customer-managed) to keep it concise
    response = iam.list_policies(Scope='Local')
    
    # Extract just the policy name from each policy dictionary
    result = [policy['PolicyName'] for policy in response['Policies']]
    
    return result

custom_policies = list_custom_iam_policies()
print("Custom IAM Policies:", custom_policies)

Custom IAM Policies: ['boto3-S3logs-mohank-1234-Policy', 'boto3-S3logs-None-Policy', 'boto3-S3logs-mohank-Policy']


In [4]:
import boto3
import json
from botocore.exceptions import ClientError

# We dynamically construct your bucket name based on the ID you provided earlier
TARGET_BUCKET = os.getenv("AWS_TARGET_BUCKET")  # Get the target bucket name from the .env file

s3_client = boto3.client('s3', region_name=os.getenv("AWS_REGION"))

try:
    # Step 1: Disable Block Public Access using Boto3
    s3_client.put_public_access_block(
        Bucket=TARGET_BUCKET,
        PublicAccessBlockConfiguration={
            'BlockPublicAcls': False,        # Allow public ACLs
            'IgnorePublicAcls': False,       # Don't ignore public ACLs
            'BlockPublicPolicy': False,      # IMPORTANT - Allow policies
            'RestrictPublicBuckets': False   # Allow public bucket
        }
    )
    print("INFO: ✓ Disabled Block Public Access via Boto3")

    # Step 2: Create and attach public read policy
    bucket_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Principal": "*",
                "Action": "s3:GetObject",
                "Resource": f"arn:aws:s3:::{TARGET_BUCKET}/*"
            }
        ]
    }

    s3_client.put_bucket_policy(
        Bucket=TARGET_BUCKET,
        Policy=json.dumps(bucket_policy)
    )
    print("INFO: ✓ Policy attached successfully to", TARGET_BUCKET)

except ClientError as e:
    print("ERROR:", e)

INFO: ✓ Disabled Block Public Access via Boto3
INFO: ✓ Policy attached successfully to phase2-s3-pyboto3-bucket-mohank-1234


In [5]:
import boto3

# Function to list all S3 bucket names
def list_s3_buckets():
    result = []
    # Initialize the S3 client
    s3 = boto3.client('s3')
    
    # Call list_buckets()
    response = s3.list_buckets()
    
    # Extract just the name from each bucket dictionary
    result = [bucket['Name'] for bucket in response['Buckets']]
    
    return result

bucket_names = list_s3_buckets()
print("My S3 Buckets:", bucket_names)

My S3 Buckets: ['phase2-s3-pyboto3-bucket-mohank-1234']


In [6]:
import time
logs_client = boto3.client('logs', region_name=os.getenv("AWS_REGION"))

LOG_GROUP = "/aws/custom/s3-utility-logs"
LOG_STREAM = "ExecutionStream"

def setup_cloudwatch_logs():
    """Create Log Group and Stream in CloudWatch if they don't exist."""
    try:
        logs_client.create_log_group(logGroupName=LOG_GROUP)
    except ClientError: pass # Ignore if it already exists

    try:
        logs_client.create_log_stream(logGroupName=LOG_GROUP, logStreamName=LOG_STREAM)
    except ClientError: pass

def send_log_to_cloudwatch(message):
    """Print locally and send log message to AWS CloudWatch Logs."""
    print(f"INFO: {message}")
    timestamp = int(round(time.time() * 1000))
    try:
        logs_client.put_log_events(
            logGroupName=LOG_GROUP,
            logStreamName=LOG_STREAM,
            logEvents=[{'timestamp': timestamp, 'message': message}]
        )
    except Exception as e:
        print(f"Failed to send log to CloudWatch: {e}")

setup_cloudwatch_logs()
send_log_to_cloudwatch("CloudWatch setup complete. Ready for pipeline execution.")

INFO: CloudWatch setup complete. Ready for pipeline execution.


In [16]:
import os

files = [
    "data/clickstream_day1.json",
    "data/clickstream_day2.json",
    "data/customers.csv",
    "data/order_items_day1.json",
    "data/order_items_day2.json",
    "data/orders_day1.json",
    "data/orders_day2.json",
    "data/products.csv"
]  # Wrap the path in a list to iterate over it

In [17]:
def upload_file_to_s3(files, target_bucket, object_name):
    """Uploads the dataset to your S3 bucket."""
    try:
        for file in files:
            s3_client.upload_file(file, target_bucket, f"raw/{file}")
            send_log_to_cloudwatch(f"File successfully uploaded to {target_bucket}/{object_name}")
    except ClientError as e:
        send_log_to_cloudwatch(f"Upload failed: {e}")

s3_destination_key = "raw/"

# Execute the upload!
upload_file_to_s3(files, TARGET_BUCKET, s3_destination_key)

INFO: File successfully uploaded to phase2-s3-pyboto3-bucket-mohank-1234/raw/
INFO: File successfully uploaded to phase2-s3-pyboto3-bucket-mohank-1234/raw/
INFO: File successfully uploaded to phase2-s3-pyboto3-bucket-mohank-1234/raw/
INFO: File successfully uploaded to phase2-s3-pyboto3-bucket-mohank-1234/raw/
INFO: File successfully uploaded to phase2-s3-pyboto3-bucket-mohank-1234/raw/
INFO: File successfully uploaded to phase2-s3-pyboto3-bucket-mohank-1234/raw/
INFO: File successfully uploaded to phase2-s3-pyboto3-bucket-mohank-1234/raw/
INFO: File successfully uploaded to phase2-s3-pyboto3-bucket-mohank-1234/raw/


In [18]:
def list_s3_objects(target_bucket, prefix=""):
    """Lists objects in the S3 bucket to verify the upload."""
    try:
        response = s3_client.list_objects_v2(Bucket=target_bucket, Prefix=prefix)
        if 'Contents' in response:
            for obj in response['Contents']:
                send_log_to_cloudwatch(f"Found object in S3: {obj['Key']} (Size: {obj['Size']} bytes)")
        else:
            send_log_to_cloudwatch("No objects found with the specified prefix.")
    except ClientError as e:
        send_log_to_cloudwatch(f"Listing objects failed: {e}")

# Verify listing in the 'uploads/' folder
list_s3_objects(TARGET_BUCKET, prefix="raw/data/")

send_log_to_cloudwatch("S3 Utility script execution completed successfully.")

INFO: Found object in S3: raw/data/clickstream_day1.json (Size: 1042717 bytes)
INFO: Found object in S3: raw/data/clickstream_day2.json (Size: 1043095 bytes)
INFO: Found object in S3: raw/data/customers.csv (Size: 201522 bytes)
INFO: Found object in S3: raw/data/order_items_day1.json (Size: 5429112 bytes)
INFO: Found object in S3: raw/data/order_items_day2.json (Size: 1344439 bytes)
INFO: Found object in S3: raw/data/orders_day1.json (Size: 2464934 bytes)
INFO: Found object in S3: raw/data/orders_day2.json (Size: 587018 bytes)
INFO: Found object in S3: raw/data/products.csv (Size: 19314 bytes)
INFO: S3 Utility script execution completed successfully.
